# 3 — Standing on the Shoulders of Giants: Fine-tuning EEG Foundation Models

**Cutting-EEG Workshop 2026 · UCSD · Deep Learning EEG Methods and Practice**

Notebook 2 ended on a problem: Transformers need more data than a typical EEG experiment
provides. Foundation models are the field's answer. Train once on thousands of hours of
unlabeled EEG, then adapt to your 280-trial study.

This is the notebook closest to how you'd actually use deep learning in a lab, because
almost nobody has enough labeled data to train from scratch.

### What we do here

1. **Self-supervised pretraining** — learn from unlabeled EEG via masked reconstruction
2. **Transfer** — move the encoder to a new subject with a fresh classification head
3. **Compare three strategies**: linear probe, full fine-tune, from-scratch
4. **Label efficiency** — the curve that actually justifies the whole approach
5. Where to get real published checkpoints, and how to load them

### An important note on scope

We pretrain a **small model on other subjects from this dataset**, inside the notebook, in
a few minutes. Real foundation models (LaBraM, BENDR, EEGPT, Brant) are pretrained on
thousands of hours across many datasets.

We do it this way on purpose: the *mechanics* — masking, transfer, freezing, label
efficiency — are identical, and this way the notebook runs end-to-end in ~10 minutes with
no external downloads and no checkpoint that might have moved. Section 7 shows exactly how
to swap in a real checkpoint.

---
**Runtime: `Runtime → Change runtime type → T4 GPU`** (strongly recommended here).

In [ ]:
%pip install -q "braindecode[moabb]"

import IPython
print("Install finished — restarting the runtime.")
print("This 'crash' notice is expected. Continue at Section 1 below.")
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import mne
import matplotlib.pyplot as plt
from copy import deepcopy

mne.set_log_level("ERROR")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 20260916
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"torch {torch.__version__} · device {DEVICE}")
if DEVICE == "cpu":
    print("CPU detected — this notebook is much happier on a GPU (Runtime → T4).")

## 1 · The setup that makes transfer meaningful

The experiment only means something if pretraining and fine-tuning use **disjoint
subjects**. Otherwise the encoder has already seen the test subject and we're measuring
memorization.

- **Pretrain** on subjects 1, 2, 4, 5, 6 — *labels discarded*
- **Fine-tune + evaluate** on subject 3 — held out entirely

This mirrors the real situation: lots of EEG exists somewhere; your new participant is
new.

In [ ]:
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import (
    Preprocessor, preprocess, exponential_moving_standardize,
    create_windows_from_events,
)
from torch.utils.data import DataLoader, TensorDataset

PRETRAIN_SUBJECTS = [1, 2, 4, 5, 6]
TARGET_SUBJECT = 3

PREPROCESSORS = [
    Preprocessor("pick_types", eeg=True, meg=False, eog=False),
    Preprocessor(lambda d: d * 1e6),
    Preprocessor("filter", l_freq=4.0, h_freq=38.0),
    Preprocessor(exponential_moving_standardize, factor_new=1e-3, init_block_size=1000),
]


def load_windows(subject_ids):
    """Load → preprocess → window, identically for every subject."""
    ds = MOABBDataset(dataset_name="BNCI2014_001", subject_ids=subject_ids)
    preprocess(ds, PREPROCESSORS, n_jobs=1)
    sfreq = ds.datasets[0].raw.info["sfreq"]
    return create_windows_from_events(
        ds,
        trial_start_offset_samples=int(-0.5 * sfreq),
        trial_stop_offset_samples=0,
        preload=True,
    ), sfreq


print(f"Loading pretraining subjects {PRETRAIN_SUBJECTS} ...")
pretrain_windows, sfreq = load_windows(PRETRAIN_SUBJECTS)

print(f"Loading target subject {TARGET_SUBJECT} ...")
target_windows, _ = load_windows([TARGET_SUBJECT])

X0, _, _ = target_windows[0]
n_channels, n_times = X0.shape
class_names = target_windows.datasets[0].windows.event_id
n_classes = len(class_names)

print(f"\nPretrain pool : {len(pretrain_windows)} windows (labels will be discarded)")
print(f"Target subject: {len(target_windows)} windows")
print(f"Shape: {n_channels} channels × {n_times} samples · {n_classes} classes")

In [ ]:
# Pretraining data: X only. Throwing the labels away is the point of self-supervision.
X_pre = torch.stack([torch.as_tensor(pretrain_windows[i][0]) for i in range(len(pretrain_windows))]).float()
pretrain_loader = DataLoader(TensorDataset(X_pre), batch_size=64, shuffle=True, drop_last=True)

# Target subject: proper session split, exactly as in Notebook 1
splits = target_windows.split("session")
train_set, test_set = splits["0train"], splits["1test"]

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, drop_last=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

print(f"Unlabeled pretraining tensor: {tuple(X_pre.shape)}")
print(f"Target train/test: {len(train_set)} / {len(test_set)} windows")

## 2 · The pretext task: masked reconstruction

How do you learn from data with no labels? Invent a task the data answers itself.

**Mask a random 50% of the patches and make the model reconstruct them.** To fill in a
missing 100 ms, the model must learn how EEG behaves — its rhythms, its spatial
correlations, how activity evolves. That knowledge lives in the encoder afterwards, and
it transfers.

This is masked language modeling (BERT) applied to EEG, and it is what LaBraM, BENDR, and
EEGPT all do, modulo details.

```
    ┌────────── PRETRAIN (unlabeled, subjects 1,2,4,5,6) ──────────┐
    EEG → patches → mask 50% → Encoder → Decoder → reconstruct
                                  │
                                  │  keep the encoder, throw away the decoder
                                  ▼
    ┌────────── FINE-TUNE (labeled, subject 3) ────────────────────┐
    EEG → patches → Encoder → new classification head → 4 classes
```

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, n_channels, patch_size, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv1d(n_channels, embed_dim, patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).transpose(1, 2)


class EEGEncoder(nn.Module):
    """The transferable part — patch embedding + Transformer encoder.

    Deliberately has NO task head. It maps EEG to per-patch representations, and both
    the pretext task and the downstream classifier attach their own head on top.
    """

    def __init__(self, n_channels, n_times, patch_size=25,
                 embed_dim=64, depth=4, n_heads=4, dropout=0.1):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.n_patches = n_times // patch_size

        self.patch_embed = PatchEmbedding(n_channels, patch_size, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches, embed_dim))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.mask_token, std=0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim * 2,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, mask=None):
        """mask: (batch, n_patches) bool, True = hidden. None = encode everything."""
        x = self.patch_embed(x)
        if mask is not None:
            x = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(x), x)
        x = x + self.pos_embed
        return self.norm(self.encoder(x))


class MaskedReconstructionModel(nn.Module):
    """Encoder + lightweight decoder. Only used during pretraining."""

    def __init__(self, encoder, n_channels, patch_size):
        super().__init__()
        self.encoder = encoder
        self.decoder = nn.Sequential(
            nn.Linear(encoder.embed_dim, 128), nn.GELU(),
            nn.Linear(128, n_channels * patch_size),
        )
        self.n_channels, self.patch_size = n_channels, patch_size

    def forward(self, x, mask):
        z = self.encoder(x, mask)
        pred = self.decoder(z)
        B, P, _ = pred.shape
        return pred.view(B, P, self.n_channels, self.patch_size)


PATCH_SIZE, EMBED_DIM, DEPTH = 25, 64, 4

encoder = EEGEncoder(n_channels, n_times, PATCH_SIZE, EMBED_DIM, DEPTH).to(DEVICE)
pretrain_model = MaskedReconstructionModel(encoder, n_channels, PATCH_SIZE).to(DEVICE)

print(f"Encoder    : {sum(p.numel() for p in encoder.parameters()):,} parameters")
print(f"+ decoder  : {sum(p.numel() for p in pretrain_model.parameters()):,} total")
print(f"Patches per window: {encoder.n_patches}")

In [ ]:
def to_patches(x, patch_size):
    """(B, C, T) -> (B, n_patches, C, patch_size). The reconstruction target."""
    B, C, T = x.shape
    n_patches = T // patch_size
    return x[:, :, :n_patches * patch_size].view(B, C, n_patches, patch_size).permute(0, 2, 1, 3)


def make_mask(B, n_patches, ratio, device):
    """Random boolean mask, exactly `ratio` of patches True per sample."""
    n_masked = int(n_patches * ratio)
    noise = torch.rand(B, n_patches, device=device)
    idx = noise.argsort(dim=1)[:, :n_masked]
    mask = torch.zeros(B, n_patches, dtype=torch.bool, device=device)
    return mask.scatter(1, idx, True)


# Sanity check
_x = torch.randn(2, n_channels, n_times, device=DEVICE)
_m = make_mask(2, encoder.n_patches, 0.5, DEVICE)
print(f"patches: {tuple(to_patches(_x, PATCH_SIZE).shape)}  (B, P, C, patch)")
print(f"mask   : {tuple(_m.shape)}, {_m[0].sum().item()}/{encoder.n_patches} hidden")

## 3 · Pretraining

Loss is MSE **on the masked patches only**. Scoring visible patches would reward copying
the input, which teaches nothing.

Note what's absent: no labels anywhere. This loop would run identically on any EEG you
have lying around — resting state, a failed pilot, someone else's public dataset.

In [ ]:
from torch.optim import AdamW
import time

PRETRAIN_EPOCHS = 40
MASK_RATIO = 0.5

opt_pre = AdamW(pretrain_model.parameters(), lr=1e-3, weight_decay=0.05)
sched_pre = torch.optim.lr_scheduler.CosineAnnealingLR(opt_pre, T_max=PRETRAIN_EPOCHS)

pretrain_losses = []
start = time.time()
print(f"Pretraining {PRETRAIN_EPOCHS} epochs on {len(X_pre)} unlabeled windows\n")

for epoch in range(PRETRAIN_EPOCHS):
    pretrain_model.train()
    epoch_loss, n_batches = 0.0, 0

    for (X,) in pretrain_loader:
        X = X.to(DEVICE)
        mask = make_mask(X.shape[0], encoder.n_patches, MASK_RATIO, DEVICE)

        pred = pretrain_model(X, mask)
        target = to_patches(X, PATCH_SIZE)

        # Masked positions only.
        m = mask.unsqueeze(-1).unsqueeze(-1)
        loss = ((pred - target) ** 2 * m).sum() / (m.sum() * n_channels * PATCH_SIZE + 1e-8)

        opt_pre.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(pretrain_model.parameters(), 1.0)
        opt_pre.step()

        epoch_loss += loss.item(); n_batches += 1

    sched_pre.step()
    pretrain_losses.append(epoch_loss / n_batches)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"epoch {epoch+1:3d}/{PRETRAIN_EPOCHS} | recon loss {pretrain_losses[-1]:.4f}")

print(f"\nPretraining done in {time.time()-start:.0f}s")
print(f"Loss {pretrain_losses[0]:.4f} → {pretrain_losses[-1]:.4f} "
      f"({100*(1-pretrain_losses[-1]/pretrain_losses[0]):.0f}% reduction)")

# This is our "foundation model" checkpoint.
PRETRAINED_STATE = deepcopy(encoder.state_dict())
torch.save(PRETRAINED_STATE, "eeg_encoder_pretrained.pt")
print("Saved → eeg_encoder_pretrained.pt")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))

ax1.plot(pretrain_losses, lw=2, color="darkgreen")
ax1.set_xlabel("epoch"); ax1.set_ylabel("masked reconstruction MSE")
ax1.set_title("Pretraining loss"); ax1.grid(alpha=0.3)

# Show a reconstruction: does it actually fill in masked regions sensibly?
pretrain_model.eval()
with torch.no_grad():
    Xs = X_pre[:1].to(DEVICE)
    ms = make_mask(1, encoder.n_patches, MASK_RATIO, DEVICE)
    pr = pretrain_model(Xs, ms)
    tg = to_patches(Xs, PATCH_SIZE)

ch = 7
true_sig = tg[0, :, ch, :].cpu().numpy().flatten()
pred_sig = pr[0, :, ch, :].cpu().numpy().flatten()
t = np.arange(len(true_sig)) / sfreq

ax2.plot(t, true_sig, label="true", lw=1.4, color="black")
ax2.plot(t, pred_sig, label="reconstructed", lw=1.4, color="tab:red", alpha=0.8)
for p in torch.where(ms[0])[0].cpu().numpy():
    ax2.axvspan(p * PATCH_SIZE / sfreq, (p + 1) * PATCH_SIZE / sfreq,
                color="tab:blue", alpha=0.12)
ax2.set_xlabel("time (s)"); ax2.set_ylabel("µV (standardized)")
ax2.set_title(f"Channel {ch} — blue spans were hidden from the model")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("Reconstructions are smooth and low-amplitude — the model learns dominant rhythms,")
print("not fine detail. That's typical, and it's enough for the features to transfer.")

## 4 · Three transfer strategies

Now the practical question: **you have a pretrained encoder and a small labeled dataset.
What do you do with it?**

| Strategy | What trains | When to use |
|---|---|---|
| **Linear probe** | Head only; encoder frozen | Very little data; also a clean test of representation quality |
| **Full fine-tune** | Everything, small LR | The usual default when you have a few hundred+ trials |
| **From scratch** | Everything, random init | The baseline that tells you if pretraining did anything |

The third isn't a strategy, it's the control. Without it you cannot claim pretraining
helped.

In [ ]:
class EEGClassifierModel(nn.Module):
    """Encoder + mean-pooled classification head."""

    def __init__(self, encoder, n_classes, dropout=0.3):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(encoder.embed_dim, n_classes),
        )

    def forward(self, x):
        return self.head(self.encoder(x).mean(dim=1))   # pool over patches


def build_model(pretrained: bool, freeze_encoder: bool):
    enc = EEGEncoder(n_channels, n_times, PATCH_SIZE, EMBED_DIM, DEPTH)
    if pretrained:
        enc.load_state_dict(PRETRAINED_STATE)
    if freeze_encoder:
        for p in enc.parameters():
            p.requires_grad = False
    return EEGClassifierModel(enc, n_classes).to(DEVICE)


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n_correct, n_total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for X, y, _ in loader:
            X, y = X.to(DEVICE).float(), y.to(DEVICE).long()
            logits = model(X)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                optimizer.step()
            total_loss += loss.item() * len(y)
            n_correct += (logits.argmax(1) == y).sum().item()
            n_total += len(y)
    return total_loss / n_total, n_correct / n_total


def train_strategy(name, model, lr, n_epochs=60, tr_loader=None, te_loader=None):
    tr_loader = tr_loader or train_loader
    te_loader = te_loader or test_loader
    params = [p for p in model.parameters() if p.requires_grad]
    opt = AdamW(params, lr=lr, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)

    curve = []
    for _ in range(n_epochs):
        run_epoch(model, tr_loader, crit, opt)
        _, te_acc = run_epoch(model, te_loader, crit, None)
        sched.step(); curve.append(te_acc)

    n_train = sum(p.numel() for p in params)
    print(f"{name:<22} best {max(curve):>6.1%} | final {curve[-1]:>6.1%} | "
          f"{n_train:>9,} trainable params")
    return curve

In [ ]:
N_FT_EPOCHS = 60
print(f"{'Strategy':<22}{'best':>11}{'final':>9}{'trainable':>20}")
print("-" * 62)

torch.manual_seed(SEED)
curve_probe = train_strategy("Linear probe", build_model(True, True), lr=3e-3, n_epochs=N_FT_EPOCHS)

torch.manual_seed(SEED)
curve_finetune = train_strategy("Full fine-tune", build_model(True, False), lr=5e-4, n_epochs=N_FT_EPOCHS)

torch.manual_seed(SEED)
curve_scratch = train_strategy("From scratch", build_model(False, False), lr=1e-3, n_epochs=N_FT_EPOCHS)

### Why the learning rates differ

**Full fine-tuning uses a much lower LR (5e-4) than from-scratch (1e-3).** This is the
most common fine-tuning mistake: pretrained weights are already good, and a large
learning rate destroys them in the first few batches — *catastrophic forgetting*. You
paid for that pretraining; don't erase it in epoch 1.

Rule of thumb: fine-tune at 2–10× below your from-scratch rate.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5))
ep = range(1, N_FT_EPOCHS + 1)
ax.plot(ep, curve_scratch, lw=2, label="From scratch (no pretraining)", color="tab:red")
ax.plot(ep, curve_probe, lw=2, label="Linear probe (frozen encoder)", color="tab:orange")
ax.plot(ep, curve_finetune, lw=2, label="Full fine-tune", color="tab:green")
ax.axhline(1/n_classes, ls="--", c="gray", label="chance")
ax.set_xlabel("epoch"); ax.set_ylabel("test accuracy")
ax.set_title(f"Transfer strategies · target subject {TARGET_SUBJECT}")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Fine-tune vs scratch: {max(curve_finetune) - max(curve_scratch):+.1%}")
print(f"Probe vs scratch    : {max(curve_probe) - max(curve_scratch):+.1%}")
print("\nWith only 5 pretraining subjects the gain is modest and can vary by seed.")
print("Section 5 shows the regime where pretraining clearly pays off.")

## 5 · Label efficiency — the real argument

Comparing at full data understates pretraining. The actual promise is:
**reach usable accuracy with far fewer labels.**

This is the number that matters for your grant. Not "1% better on a benchmark" but
"we can run 20 trials per participant instead of 300."

We retrain at 10%, 25%, 50%, and 100% of the target subject's training labels.

In [ ]:
from torch.utils.data import Subset

FRACTIONS = [0.1, 0.25, 0.5, 1.0]
N_EPOCHS_LC = 50

rng = np.random.RandomState(SEED)
n_train_total = len(train_set)
perm = rng.permutation(n_train_total)

results = {"fine-tune": [], "scratch": [], "n_labels": []}

for frac in FRACTIONS:
    n_use = max(BATCH := 16, int(n_train_total * frac))
    subset = Subset(train_set, perm[:n_use].tolist())
    loader = DataLoader(subset, batch_size=min(64, n_use), shuffle=True, drop_last=False)

    results["n_labels"].append(n_use)
    print(f"\n--- {int(frac*100)}% of labels ({n_use} trials) ---")

    torch.manual_seed(SEED)
    c_ft = train_strategy("  fine-tune", build_model(True, False), 5e-4,
                          N_EPOCHS_LC, loader, test_loader)
    torch.manual_seed(SEED)
    c_sc = train_strategy("  scratch", build_model(False, False), 1e-3,
                          N_EPOCHS_LC, loader, test_loader)

    results["fine-tune"].append(max(c_ft))
    results["scratch"].append(max(c_sc))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
n_lab = results["n_labels"]

ax.plot(n_lab, results["fine-tune"], "o-", lw=2.5, ms=9,
        label="Pretrained + fine-tuned", color="tab:green")
ax.plot(n_lab, results["scratch"], "s-", lw=2.5, ms=9,
        label="From scratch", color="tab:red")
ax.axhline(1/n_classes, ls="--", c="gray", label="chance")
ax.fill_between(n_lab, results["scratch"], results["fine-tune"],
                where=[f >= s for f, s in zip(results["fine-tune"], results["scratch"])],
                alpha=0.15, color="tab:green", label="pretraining gain")
ax.set_xlabel("number of labeled training trials")
ax.set_ylabel("best test accuracy")
ax.set_title("Label efficiency — the case for foundation models")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"{'labels':>8}{'fine-tune':>12}{'scratch':>10}{'gain':>9}")
print("-" * 40)
for n, ft, sc in zip(n_lab, results["fine-tune"], results["scratch"]):
    print(f"{n:>8}{ft:>11.1%}{sc:>10.1%}{ft-sc:>+9.1%}")

print("\nThe gap is usually widest on the LEFT. That is the whole point:")
print("pretraining buys you the most exactly when labels are scarce.")

## 6 · Partial freezing

Between "freeze everything" and "train everything" there's a middle path: freeze the
early layers, fine-tune the late ones.

The intuition, borrowed from vision: early layers learn generic features (rhythms,
spatial patterns) that transfer across subjects; late layers learn task-specific ones
that should adapt. Freezing early layers cuts overfitting and trains faster.

In [ ]:
def build_partial(n_frozen_layers):
    enc = EEGEncoder(n_channels, n_times, PATCH_SIZE, EMBED_DIM, DEPTH)
    enc.load_state_dict(PRETRAINED_STATE)

    for p in enc.patch_embed.parameters():
        p.requires_grad = False
    enc.pos_embed.requires_grad = False
    for i in range(n_frozen_layers):
        for p in enc.encoder.layers[i].parameters():
            p.requires_grad = False

    return EEGClassifierModel(enc, n_classes).to(DEVICE)


print(f"{'Frozen layers':<22}{'best':>11}{'final':>9}{'trainable':>20}")
print("-" * 62)
freeze_curves = {}
for n_frozen in [0, 2, 4]:
    torch.manual_seed(SEED)
    label = f"patch+pos + {n_frozen}/{DEPTH}"
    freeze_curves[n_frozen] = train_strategy(label, build_partial(n_frozen), 5e-4, N_FT_EPOCHS)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
for n_frozen, curve in freeze_curves.items():
    ax.plot(range(1, len(curve) + 1), curve, lw=2,
            label=f"{n_frozen}/{DEPTH} encoder layers frozen")
ax.plot(range(1, len(curve_finetune) + 1), curve_finetune, lw=2, ls=":",
        color="black", label="full fine-tune (nothing frozen)")
ax.axhline(1/n_classes, ls="--", c="gray")
ax.set_xlabel("epoch"); ax.set_ylabel("test accuracy")
ax.set_title("How much to freeze?"); ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Rule of thumb: the less labeled data you have, the more you should freeze.")

## 7 · Using a real foundation model

Everything above used an encoder we pretrained in-notebook. Swapping in a published
checkpoint changes **one thing: where `state_dict` comes from.** The transfer logic,
the LR choice, the freezing decisions — all identical.

### Models worth knowing (as of 2026)

| Model | Pretraining data | Notes |
|---|---|---|
| **LaBraM** | ~2,500 h, many datasets | Vector-quantized neural tokenizer; strong multi-task results |
| **BENDR** | TUH EEG Corpus (~1,500 h) | wav2vec-style contrastive; among the first for EEG |
| **EEGPT** | Mixed public corpora | Dual self-supervised objective |
| **Brant** | Intracranial | For iEEG/SEEG, not scalp |

Most publish weights on GitHub or HuggingFace. **Check the license before you use one in
a paper or product** — they vary, and some are research-only.

### The pattern

```python
# 1. Get the architecture (usually from the authors' repo)
model = LaBraM(n_channels=22, n_classes=4)

# 2. Load published weights
state = torch.load("labram_base.pth", map_location="cpu")
state = state.get("model", state.get("state_dict", state))   # unwrap common formats

# 3. Load non-strictly — their head won't match your task
missing, unexpected = model.load_state_dict(state, strict=False)
print("missing :", missing)      # expect your new head here
print("unexpected:", unexpected) # expect their old head here

# 4. From here on, identical to this notebook
```

`strict=False` is the crucial bit, and the printouts are how you verify it worked.

In [ ]:
# Demonstrating the real-checkpoint workflow using the encoder we saved earlier.
# Everything here is exactly what you'd do with a downloaded checkpoint.

checkpoint = torch.load("eeg_encoder_pretrained.pt", map_location="cpu")
print(f"Checkpoint has {len(checkpoint)} tensors")
print("First few keys:", list(checkpoint.keys())[:4], "...\n")

fresh = EEGClassifierModel(
    EEGEncoder(n_channels, n_times, PATCH_SIZE, EMBED_DIM, DEPTH), n_classes)

# Published checkpoints store the encoder alone; our model nests it under "encoder."
prefixed = {f"encoder.{k}": v for k, v in checkpoint.items()}
missing, unexpected = fresh.load_state_dict(prefixed, strict=False)

print(f"missing keys ({len(missing)}):  {missing}")
print(f"unexpected keys ({len(unexpected)}): {unexpected}")
print("\nOnly the classification head is missing — correct. It's new and task-specific.")
print("An empty 'missing' list would mean you loaded a head trained for someone")
print("else's task; a huge list means the architectures don't match.")

### Two failure modes to recognize

**Different channel montage.** A model pretrained on 64 channels won't accept your 22.
Options: re-reference/interpolate to their montage, or replace the input projection and
retrain just that layer. Some models (LaBraM) handle variable montages by design.

**Different sampling rate.** A 200 Hz model on 250 Hz data sees signals stretched in
time — the frequency content shifts. Resample your data to match, always.

Both errors are silent: the code runs, the accuracy is quietly poor. Check them
explicitly.

## Recap

The transfer learning recipe, end to end:

1. **Pretrain** on unlabeled EEG (or download someone's checkpoint)
2. **Discard** the pretext head; keep the encoder
3. **Attach** a fresh head for your task
4. **Choose** a strategy — probe / partial freeze / full fine-tune, less data → freeze more
5. **Lower the LR** — 2–10× below from-scratch, or you erase what you loaded
6. **Always** run the from-scratch control

Key numbers from this run:

In [ ]:
print(f"Target subject {TARGET_SUBJECT}, {len(train_set)} labeled training trials\n")
print(f"{'From scratch':<28}{max(curve_scratch):>8.1%}")
print(f"{'Linear probe':<28}{max(curve_probe):>8.1%}")
print(f"{'Full fine-tune':<28}{max(curve_finetune):>8.1%}")
print(f"\nAt 10% of labels ({results['n_labels'][0]} trials):")
print(f"{'  from scratch':<28}{results['scratch'][0]:>8.1%}")
print(f"{'  pretrained':<28}{results['fine-tune'][0]:>8.1%}")
print("\nNote: our 'foundation model' saw 5 subjects. Published ones see thousands")
print("of hours. Read these gains as a lower bound on what's achievable.")

### Try it yourself

1. Add subjects 7, 8, 9 to `PRETRAIN_SUBJECTS`. Does more pretraining data help
   downstream? This is the scaling hypothesis, testable in 10 minutes.
2. Set `MASK_RATIO = 0.75`. Harder pretext task → better features, or too hard?
3. Change `TARGET_SUBJECT`. Transfer gains vary a lot by subject — which is itself a
   finding worth understanding.

**Next:** Notebook 4 adds a second modality — aligning eye-gaze with EEG and fusing them.